# PaperMind v4 — Pinecone metadata patch (B-012)

**Hedef:** `papers-bgem3` index'ine 8 metadata alanı yapıştır (D, F, S, year, q_weak, method, lang, v_conf).

**Tek zorunlu Secret:** `PINECONE_API_KEY` (Colab Secrets sekmesi)
Diğer değerler hardcoded — gerekirse Cell 1'de elle düzelt.

**Sıra:** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8. Manuel koş, "Run all" yasak.

**Toplam süre:** ~4-5 saat (join 30 dk + GCS 1h + Pinecone import 2-3h + smoke 5 dk).


## Cell 1 — Setup (Drive + pip + sabitler + sanity check)

In [ ]:
from google.colab import drive, userdata
from google.colab.userdata import SecretNotFoundError, NotebookAccessError, TimeoutException
drive.mount('/content/drive')

import subprocess
print('Installing packages (sessiz)...')
subprocess.run(['pip', '-q', 'install',
    'pinecone==8.1.2', 'polars==1.35.2', 'pyarrow==17.0.0',
    'google-cloud-storage==2.18.0'
], check=True)

import os, json, math, time, pickle, sys
from datetime import datetime, timedelta
from pathlib import Path
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
from pinecone import Pinecone
from getpass import getpass

# Hardcoded sabitler (B-011 dogrulandi, degistirme)
PINECONE_INDEX_NAME = 'papers-bgem3'
GCS_BUCKET          = 'embeddings_v3'
GCS_PREFIX          = 'papers_with_metadata_v1'

# Secret okuma - timeout/access hatasi durumunda manuel fallback
def _get_secret(key, prompt_label=None):
    try:
        v = userdata.get(key)
        return v if v else None
    except SecretNotFoundError:
        print(f'  Secret "{key}" tanimli degil.')
        return None
    except (NotebookAccessError, TimeoutException) as e:
        print(f'  Secret "{key}" Colab API hata: {type(e).__name__}')
        return None
    except Exception as e:
        print(f'  Secret "{key}" beklenmedik hata: {type(e).__name__}: {e}')
        return None

print('=== Secret okuma ===')
PINECONE_API_KEY = _get_secret('PINECONE_API_KEY')
PINECONE_INTEGRATION_ID = _get_secret('PINECONE_INTEGRATION_ID')

# Manuel fallback - Secrets calismiyorsa prompt
if not PINECONE_API_KEY:
    print('\nPINECONE_API_KEY Secrets\'ten alinamadi.')
    print('Cozum 1: Colab sol panel anahtar ikonu -> PINECONE_API_KEY -> "Notebook access" toggle ON, sonra Cell 1 tekrar kos.')
    print('Cozum 2 (simdi): asagidaki prompta key\'i yapistir (pcsk_ ile baslayan).')
    PINECONE_API_KEY = getpass('PINECONE_API_KEY: ').strip() or None

assert PINECONE_API_KEY, 'PINECONE_API_KEY hala yok - durdu.'
assert PINECONE_API_KEY.startswith('pcsk_'), f'API key format yanlis (pcsk_ ile baslamali): {PINECONE_API_KEY[:10]}...'

# Drive paths
DRIVE_ROOT = Path('/content/drive/MyDrive/Dataleak')
PATHS = {
    'paper_card':    DRIVE_ROOT / 'facts/fact_paper_id_card.parquet',
    'paper_field':   DRIVE_ROOT / 'facts/fact_paper_field.parquet',
    'paper_quality': DRIVE_ROOT / 'facts/fact_paper_quality_v3.parquet',
    'paper_method':  DRIVE_ROOT / 'facts/fact_paper_metod.parquet',
    'embed_a':       DRIVE_ROOT / 'dims/fact_paper_embedding_bge_m3_shard_a.parquet',
    'embed_b':       DRIVE_ROOT / 'dims/fact_paper_embedding_bge_m3_shard_b.parquet',
}
WORK_ROOT = Path('/content/pinecone_metadata_work')
WORK_ROOT.mkdir(exist_ok=True)

print('\n=== Drive parquet kontrol ===')
all_ok = True
for name, p in PATHS.items():
    exists = p.exists()
    size_gb = p.stat().st_size / 1e9 if exists else 0
    flag = '+' if exists else 'X MISSING'
    if not exists: all_ok = False
    print(f'  {flag}  {name:15s}  {p.name}  ({size_gb:.2f} GB)')
assert all_ok, 'Bir parquet eksik - dur.'

print('\n=== Sabitler ===')
print(f'  PINECONE_INDEX_NAME:     {PINECONE_INDEX_NAME}')
print(f'  PINECONE_API_KEY:        + (***{PINECONE_API_KEY[-4:]})')
print(f'  PINECONE_INTEGRATION_ID: {PINECONE_INTEGRATION_ID or "(yok - Cell 6 public-bucket dener)"}')
print(f'  GCS_BUCKET:              {GCS_BUCKET}')
print(f'  GCS_PREFIX:              {GCS_PREFIX}')

pc = Pinecone(api_key=PINECONE_API_KEY)
try:
    idx_info = pc.describe_index(PINECONE_INDEX_NAME)
except Exception as e:
    print(f'\nX Pinecone describe_index hata: {e}')
    print('Mevcut indexler:')
    for i in pc.list_indexes():
        print(f'  - {i.name}')
    raise

print(f'\n=== Pinecone ===')
print(f'  index: {idx_info.name}')
print(f'  host:  {idx_info.host}')
print(f'  dim:   {idx_info.dimension}')
print(f'  metric: {idx_info.metric}')

idx = pc.Index(PINECONE_INDEX_NAME)
stats = idx.describe_index_stats()
print(f'  current vector_count: {stats.total_vector_count:,}  (expected 24,867,210 from B-011)')

print("\n+ Cell 1 PASS - Cell 2'ye gec")


## Cell 2 — 4 parquet schema audit (envanter doğrulama)

In [ ]:
print('=== 4 metadata kaynağı schema ===')
schemas = {}
for name in ['paper_card', 'paper_field', 'paper_quality', 'paper_method']:
    pf = pq.ParquetFile(str(PATHS[name]))
    cols = pf.schema_arrow.names
    schemas[name] = cols
    print(f'\n{name} ({pf.metadata.num_rows:,} rows):')
    print(f'  cols: {cols}')

expected = {
    'paper_card':    ['paper_id', 'year', 'language', 'topic_profile'],
    'paper_field':   ['paper_id', 'primary_domain', 'primary_field', 'primary_subfield'],
    'paper_quality': ['paper_id', 'q_weak'],
    'paper_method':  ['paper_id'],
}
for name, must_have in expected.items():
    missing = [c for c in must_have if c not in schemas[name]]
    if missing:
        print(f'\n✗ {name}: kolon eksik {missing}')
    else:
        print(f'\n✓ {name}: gerekli kolonlar var')

method_col = None
for c in ('metod_id', 'method_id', 'method', 'method_name', 'method_label'):
    if c in schemas['paper_method']:
        method_col = c
        break
assert method_col, 'paper_method method kolonu bulunamadi'
print(f'\nmethod kolon adi: {method_col}')

score_col = None
for c in ('score', 'p_method_in_paper', 'weight', 'confidence'):
    if c in schemas['paper_method']:
        score_col = c
        break
print(f'method skor kolonu adi: {score_col or "(yok)"}')

has_primary = 'is_primary' in schemas['paper_method']
print(f'is_primary kolonu: {has_primary}')

print("\n+ Cell 2 PASS - Cell 3'e gec")


## Cell 3 — Polars stream join → unified metadata parquet

Çıktı şema (24.86M × 9 col, ~1 GB):
- `paper_id` String, `D/F/S` String, `year` Int32, `q_weak` Float32, `method` String, `lang` String, `v_conf` Float32


In [ ]:
META_PARQUET = WORK_ROOT / 'paper_metadata_unified.parquet'

def _norm_pid(col='paper_id'):
    return pl.col(col).str.split('/').list.last().alias(col)

print('Loading PaperCard master...')
pc_df = (
    pl.scan_parquet(str(PATHS['paper_card']))
    .with_columns(_norm_pid())
    .select([
        pl.col('paper_id'),
        pl.col('year').cast(pl.Int32),
        pl.col('language').alias('lang'),
        pl.col('topic_profile').struct.field('theme_scores').list.first().cast(pl.Float32).alias('v_conf'),
    ])
)

print('Loading paper_field...')
field_df = (
    pl.scan_parquet(str(PATHS['paper_field']))
    .with_columns(_norm_pid())
    .select([
        pl.col('paper_id'),
        pl.col('primary_domain').alias('D'),
        pl.col('primary_field').alias('F'),
        pl.col('primary_subfield').alias('S'),
    ])
)

print('Loading paper_quality...')
quality_df = (
    pl.scan_parquet(str(PATHS['paper_quality']))
    .with_columns(_norm_pid())
    .select([pl.col('paper_id'), pl.col('q_weak').cast(pl.Float32)])
)

print('Loading paper_method (group_by primary)...')
mlf = pl.scan_parquet(str(PATHS['paper_method'])).with_columns(_norm_pid())
if has_primary:
    mlf = mlf.filter(pl.col('is_primary') == True)
if score_col:
    method_df = mlf.group_by('paper_id').agg(
        pl.col(method_col).sort_by(score_col, descending=True).first().cast(pl.Utf8).alias('method')
    )
else:
    method_df = mlf.group_by('paper_id').agg(
        pl.col(method_col).first().cast(pl.Utf8).alias('method')
    )

print('Streaming join + sink_parquet...')
t0 = time.time()
unified = (
    pc_df
    .join(field_df,   on='paper_id', how='left')
    .join(quality_df, on='paper_id', how='left')
    .join(method_df,  on='paper_id', how='left')
    .select(['paper_id', 'D', 'F', 'S', 'year', 'q_weak', 'method', 'lang', 'v_conf'])
)
unified.sink_parquet(str(META_PARQUET), compression='zstd')
elapsed_str = str(timedelta(seconds=int(time.time()-t0)))
print('  + join+sink sure: ' + elapsed_str)

mpf = pq.ParquetFile(str(META_PARQUET))
n_rows = mpf.metadata.num_rows
size_mb = META_PARQUET.stat().st_size/1e6
print('\n+ unified metadata: {:,} rows, {:.1f} MB'.format(n_rows, size_mb))
print(f'  cols: {mpf.schema_arrow.names}')

assert 24_000_000 < n_rows < 25_000_000, f'Beklenen 24-25M satir, gelen {n_rows:,}'

sample_df = next(mpf.iter_batches(batch_size=5)).to_pandas()
print('\nIlk 5 satir:')
print(sample_df)

print('\nKolon doluluk (10K sample):')
sample_full = next(mpf.iter_batches(batch_size=10000)).to_pandas()
for col in sample_full.columns:
    non_null = sample_full[col].notna().sum()
    pct = 100*non_null/len(sample_full)
    print(f'  {col:10s}: {non_null:,}/{len(sample_full):,} ({pct:.1f}%)')

print("\n+ Cell 3 PASS - Cell 4'e gec")


## Cell 4 — Pinecone-ready shard parquet (`id, values, metadata`)

BGE-M3 embedding shard A + B × unified metadata inner join. Sparse drop (DM-016).
Çıktı: 2 shard parquet (~25 GB her biri, ~50 GB toplam).


In [ ]:
print('Loading meta_dict (~3 GB RAM)...')
meta_lf = pl.scan_parquet(str(META_PARQUET))
meta_df = meta_lf.collect()
meta_dict = {}
for row in meta_df.iter_rows(named=True):
    pid = row.pop('paper_id')
    meta_dict[pid] = {k: v for k, v in row.items() if v is not None}
print(f'  + meta_dict size: {len(meta_dict):,}')
del meta_df

sample_pf = pq.ParquetFile(str(PATHS['embed_a']))
embed_cols = sample_pf.schema_arrow.names
print(f'\nembed shard kolonlari: {embed_cols}')
dense_col = None
for cand in ('dense_embedding', 'embedding', 'values', 'dense', 'dense_vector', 'embedding_dense'):
    if cand in embed_cols:
        dense_col = cand
        break
assert dense_col, f'dense kolon bulunamadi: {embed_cols}'
print(f'dense kolon adi: {dense_col}')

def _bare_w(pid_str):
    s = str(pid_str).strip()
    if s.startswith('https://openalex.org/'):
        s = s[len('https://openalex.org/'):]
    return s

def build_pinecone_shard(embed_path, out_path, shard_label):
    print(f'\n=== shard {shard_label}: {embed_path.name} -> {out_path.name} ===')
    pf = pq.ParquetFile(str(embed_path))
    total = pf.metadata.num_rows
    print(f'  total rows: {total:,}')

    schema = pa.schema([
        pa.field('id', pa.string()),
        pa.field('values', pa.list_(pa.float32())),
        pa.field('metadata', pa.string()),
    ])
    writer = pq.ParquetWriter(str(out_path), schema, compression='snappy')

    written = 0
    skipped_no_meta = 0
    t0 = time.time()
    BATCH = 100_000
    for batch in pf.iter_batches(batch_size=BATCH, columns=['paper_id', dense_col]):
        df = batch.to_pandas()
        kept_ids, kept_vecs, kept_metas = [], [], []
        for pid_raw, vec in zip(df['paper_id'].tolist(), df[dense_col]):
            pid = _bare_w(pid_raw)
            md_d = meta_dict.get(pid)
            if md_d is None:
                skipped_no_meta += 1
                continue
            kept_ids.append(pid)
            kept_vecs.append([float(x) for x in vec])
            kept_metas.append(json.dumps(md_d, ensure_ascii=False, default=str))
        if kept_ids:
            tbl = pa.Table.from_pydict({
                'id': kept_ids,
                'values': kept_vecs,
                'metadata': kept_metas,
            }, schema=schema)
            writer.write_table(tbl)
            written += len(kept_ids)
        elapsed = time.time() - t0
        rate = written / elapsed if elapsed > 0 else 0
        eta = (total - written) / rate if rate > 0 else 0
        stamp = datetime.now().strftime('%H:%M:%S')
        pct = 100*written/total
        eta_s = str(timedelta(seconds=int(eta)))
        print(f'    [{stamp}] {written:>11,}/{total:,} ({pct:5.1f}%)  rate={rate:>6,.0f}/s  ETA={eta_s}', flush=True)
    writer.close()
    size_gb = out_path.stat().st_size / 1e9
    print(f'  + shard {shard_label}: written={written:,}, skipped_no_meta={skipped_no_meta:,}, size={size_gb:.2f} GB')
    return written, skipped_no_meta

OUT_A = WORK_ROOT / 'pinecone_upsert_shard_a.parquet'
OUT_B = WORK_ROOT / 'pinecone_upsert_shard_b.parquet'
written_a, skip_a = build_pinecone_shard(PATHS['embed_a'], OUT_A, 'A')
written_b, skip_b = build_pinecone_shard(PATHS['embed_b'], OUT_B, 'B')

print(f'\n=== Toplam ===')
print(f'  written: {written_a + written_b:,}')
print(f'  skipped_no_meta: {skip_a + skip_b:,}  (beklenen ~5,000 = PaperCard miss)')
total_size_gb = (OUT_A.stat().st_size + OUT_B.stat().st_size) / 1e9
print(f'  size: {total_size_gb:.2f} GB')

print("\n+ Cell 4 PASS - Cell 5'e gec")


## Cell 5 — GCS upload (`gs://embeddings_v3/papers_with_metadata_v1/`)

Authenticate sırasında GCP project otomatik seçilir. Yol B'de kullandığın bucket aynı.


In [ ]:
from google.cloud import storage
from google.colab import auth as gauth

print('Authenticate (popup açılabilir, izin ver)...')
gauth.authenticate_user()

# Default project: gauth oturumundan
client = storage.Client()
print(f'Active project: {client.project}')

bucket = client.bucket(GCS_BUCKET)
if not bucket.exists():
    raise RuntimeError(f'Bucket gs://{GCS_BUCKET} bulunamadı veya erişim yok. Pinecone Console > Storage Integrations\'da bu bucket için integration\'ı doğrula.')

print(f'\nBucket: gs://{GCS_BUCKET}/  ✓')
print(f'Hedef prefix: {GCS_PREFIX}/\n')

for local_path in [OUT_A, OUT_B]:
    blob_name = f'{GCS_PREFIX}/{local_path.name}'
    size_gb = local_path.stat().st_size / 1e9
    print(f'Uploading {local_path.name} ({size_gb:.2f} GB) → gs://{GCS_BUCKET}/{blob_name}...')
    blob = bucket.blob(blob_name)
    blob.chunk_size = 256 * 1024 * 1024  # 256 MB
    t0 = time.time()
    blob.upload_from_filename(str(local_path))
    print(f'  ✓ upload süre: {timedelta(seconds=int(time.time()-t0))}')

# Upload doğrulama — GCS'te listele
blobs = list(client.list_blobs(GCS_BUCKET, prefix=GCS_PREFIX + '/'))
print(f'\n=== gs://{GCS_BUCKET}/{GCS_PREFIX}/ içindeki dosyalar ===')
for b in blobs:
    print(f'  {b.name}  ({b.size/1e9:.2f} GB)')

print('\n✓ Cell 5 PASS — Cell 6\'ya geç')


## Cell 6 — Pinecone bulk import (upsert mode)

Mevcut 24.87M vector korunur (ID match → values aynı, metadata yapıştırılır).


In [ ]:
import_uri = f'gs://{GCS_BUCKET}/{GCS_PREFIX}/'
print(f'Import URI: {import_uri}')

if PINECONE_INTEGRATION_ID:
    print(f'Using PINECONE_INTEGRATION_ID: ***{PINECONE_INTEGRATION_ID[-4:]}')
    op = idx.start_import(uri=import_uri, integration_id=PINECONE_INTEGRATION_ID)
else:
    print('No integration_id — public bucket olarak deneniyor (B-011\'de çalıştıysa OK).')
    op = idx.start_import(uri=import_uri)

import_id = op.id if hasattr(op, 'id') else op.get('id')
print(f'\n✓ IMPORT_ID = {import_id}')
print('  Cell 7\'ye geç (polling).')


## Cell 7 — Status poll (60 s × 8 saat timeout)

In [ ]:
POLL_INTERVAL = 60
TIMEOUT_HOURS = 8

print(f'Polling every {POLL_INTERVAL} s, timeout {TIMEOUT_HOURS} h')
print('-' * 60)
t_start = time.time()
status = None
while True:
    info = idx.describe_import(import_id)
    status = info.status
    pct = float(getattr(info, 'percent_complete', 0) or 0)
    records = int(getattr(info, 'records_imported', 0) or 0)
    stamp = datetime.now().strftime('%H:%M:%S')
    print(f'[{stamp}] status={status} pct={pct:.1f} records={records:,}', flush=True)
    if status in ('Completed', 'Failed', 'Cancelled'):
        break
    if time.time() - t_start > TIMEOUT_HOURS * 3600:
        print('TIMEOUT — manuel kontrol gerek.')
        break
    time.sleep(POLL_INTERVAL)

print(f'\n=== Final ===')
print(f'  status: {status}')
print(f'  süre:   {timedelta(seconds=int(time.time()-t_start))}')
assert status == 'Completed', f'Import {status} oldu — Cell 8\'e geçmeden hatayı incele.'
print('\n✓ Cell 7 PASS — Cell 8\'e geç')


## Cell 8 — Smoke verify (vector + metadata + query latency)

In [ ]:
print('=== 1. describe_index_stats ===')
stats = idx.describe_index_stats()
print(f'  total: {stats.total_vector_count:,}')
print(f'  expected: 24,867,210  (upsert mode mevcut korur)')
delta = stats.total_vector_count - 24_867_210
v_parity = (delta == 0)
print(f'  delta: {delta:+,}  →  {"✓ parity" if v_parity else "✗ FAIL"}')

print('\n=== 2. fetch metadata audit (3 sample) ===')
sample_ids = list(meta_dict.keys())[:3]
fetched = idx.fetch(ids=sample_ids, namespace='__default__')
expected_keys = {'D', 'F', 'S', 'year', 'q_weak', 'method', 'lang', 'v_conf'}
all_ok = True
for vid, vec in fetched.vectors.items():
    md_d = dict(vec.metadata) if vec.metadata else {}
    found = set(md_d.keys())
    missing = expected_keys - found
    flag = '✓' if not missing else f'✗ missing={missing}'
    if missing: all_ok = False
    print(f'  {vid}: {flag}')
    print(f'    sample: {dict(list(md_d.items())[:5])}')

print('\n=== 3. query smoke (topK=10) ===')
qvec = list(fetched.vectors.values())[0].values if fetched.vectors else None
dt_ms = -1
if qvec:
    t0 = time.time()
    qres = idx.query(vector=qvec, top_k=10, namespace='__default__', include_metadata=True)
    dt_ms = (time.time() - t0) * 1000
    print(f'  latency: {dt_ms:.1f} ms (DoD <500 ms)')
    print(f'  matches: {len(qres.matches)}')
    for m in qres.matches[:5]:
        keys = sorted(m.metadata.keys()) if m.metadata else []
        print(f'    {m.id}  score={m.score:.4f}  meta_keys={keys}')

print('\n=== B-012 DoD ===')
print(f'  vector parity:           {"✓" if v_parity else "✗"}')
print(f'  metadata 8-field:        {"✓" if all_ok else "✗"}')
print(f'  query latency <500ms:    {"✓" if (qvec and 0 <= dt_ms < 500) else "✗"}')

if v_parity and all_ok and qvec and 0 <= dt_ms < 500:
    print('\n✓✓✓ B-012 KAPANDI — Pinecone metadata patch tam parity')
else:
    print('\n✗ B-012 EKSİK — yukarıdaki ✗ işaretlerini incele')
